<a href="https://colab.research.google.com/github/ultimog/econ3916-lab11-sql/blob/main/Econ_3916_PS3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Econ 3916 PS3 — The Number Nobody Can Reproduce

**ECON 3916 · Applied Data Analytics** · Topics 9-11 · 100 points · your role: Analyst, Business Intelligence

This is the **starter**: the structure of the set in rubric order. Fill the empty cells; keep every heading so the grader finds each step. The brief on Canvas has the full text of every step, the numbers you must hit and the rubric. Every claim goes in a text cell: the reader is analyst's audience, not a grader with a kernel.

**AI is not allowed in Phases 1–3.** Generative AI is **prohibited** in Phases 1 through 3. Write the SQL yourself; that is the skill being assessed and it is the one interviewers test. AI is authorised and required in Phase 4.

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

rng = np.random.default_rng(42)     # one generator, so every draw reruns the same way
pd.set_option('display.precision', 3)

## Phase 1: Reproduce Both Numbers (25 pts)

### Step 1.1: Build the warehouse (8 pts)

In [ ]:
!pip install -q duckdb                      # the brief's setup step 2 (quick if Colab already has it)
import duckdb
import statsmodels.formula.api as smf       # for Phase 3

con = duckdb.connect()                       # your warehouse: register customers and orders here

# Parameters the brief fixes (Step 1.1). Keep them here so the grader can rerun you.
N_CUSTOMERS = 3_000          # at least 3,000 customers ...
MIN_WEST = 2_000             # ... and at least 2,000 of them in the West
NB_N, NB_P = 0.8, 0.12       # n_orders = 1 + NegativeBinomial(n=0.8, p=0.12): mean near 7, long right tail
REV_PER_ORDER = 60           # revenue = 60 * n_orders + Normal(400, 150), floored at 50
REV_MEAN, REV_SD, REV_FLOOR = 400, 150, 50

In [ ]:
N = N_CUSTOMERS

# About 70% of customers are in the West, so the West has at least 2,000
region = rng.choice(["West", "East", "South"], size=N, p=[0.70, 0.15, 0.15])

# Order counts: 1 + NegativeBinomial(0.8, 0.12), a mean near 7 with a long right tail
n_orders = 1 + rng.negative_binomial(NB_N, NB_P, size=N)

# Revenue depends on order count, floored at 50
revenue = np.maximum(REV_PER_ORDER * n_orders + rng.normal(REV_MEAN, REV_SD, size=N), REV_FLOOR)

# Signup dates spread over three years
signup_date = pd.Timestamp("2022-01-01") + pd.to_timedelta(rng.integers(0, 1095, size=N), unit="D")

customers_df = pd.DataFrame({"customer_id": np.arange(1, N + 1),
                             "region": region,
                             "revenue": revenue.round(2),
                             "signup_date": signup_date})

# One row per order: np.repeat repeats each customer's id once per order they placed
orders_df = pd.DataFrame({"order_id": np.arange(1, n_orders.sum() + 1),
                          "customer_id": np.repeat(customers_df["customer_id"].to_numpy(), n_orders)})

assert (region == "West").sum() >= MIN_WEST, "fewer than 2,000 West customers"

# Load both tables into DuckDB
con.execute("CREATE OR REPLACE TABLE customers AS SELECT * FROM customers_df")
con.execute("CREATE OR REPLACE TABLE orders AS SELECT * FROM orders_df")

print(con.execute("SELECT COUNT(*) AS customers FROM customers").df())
print(con.execute("SELECT COUNT(*) AS orders FROM orders").df())
print(con.execute("SELECT region, COUNT(*) AS n FROM customers GROUP BY region ORDER BY region").df())
print(f"Mean orders per customer: {n_orders.mean():.2f}, largest: {n_orders.max()}")

   customers
0       3000
   orders
0   20584
  region     n
0   East   443
1  South   460
2   West  2097
Mean orders per customer: 6.86, largest: 57


I made 3,000 customers (seed 42), and 2,097 of them ended up in the West. Together they placed 20,584 orders. Most customers only order a handful of times, but a few order a lot (one placed 57). I also made revenue go up with the number of orders, on purpose, because that's the whole reason the two queries end up disagreeing.

### Step 1.2: Write both queries (7 pts)

In [ ]:
# The low number: one row per customer
customer_level = con.execute("""
    SELECT AVG(revenue) AS avg_revenue, COUNT(*) AS rows_used
    FROM customers
    WHERE region = 'West'
""").df()

# The high number: joining orders repeats each customer once per order
order_level = con.execute("""
    SELECT AVG(c.revenue) AS avg_revenue, COUNT(*) AS rows_used
    FROM customers AS c
    JOIN orders AS o ON c.customer_id = o.customer_id
    WHERE c.region = 'West'
""").df()

low, high = customer_level["avg_revenue"][0], order_level["avg_revenue"][0]
print(f"Un-joined: ${low:,.2f} over {customer_level['rows_used'][0]:,} rows")
print(f"Joined:    ${high:,.2f} over {order_level['rows_used'][0]:,} rows")
print(f"Ratio:     {high / low:.2f}")

> **🛑 Verification checkpoint.** On West-region customers you should get about
> **$812** from the un-joined query (accept $780 to $845) and about **$1,240**
> from the joined one (accept $1,160 to $1,320), a ratio between **1.45 and
> 1.60**. If the two numbers are close together, your revenue is not correlated
> with order count, there is no fan-out to find, and the rest of the problem set
> has nothing to work on — go back to Step 1.1. **Do not proceed until it
> matches.**

**Checkpoint met.** The query without the join gives \$814.43, using 2,097 rows (one per West customer). The query with the join gives \$1,250.67, using 14,191 rows (one per West order). The joined number is 1.54 times bigger.

The right number is **\$814**. The question asks for average revenue per *customer*, so each customer should count once. Once you join in the orders table, though, every customer shows up once for each order they placed. The customers who order the most also bring in the most revenue, so they get counted over and over, and that drags the average up. So the \$1,240 isn't really an average over customers. It's an average over orders that just happens to use a customer column.

### Step 1.3: The rule (10 pts)

In [ ]:
def check_join(con, before_sql, after_sql):
    """Warn if a join changes the number of rows. Returns True if the row count is unchanged."""
    n_before = con.execute(f"SELECT COUNT(*) FROM ({before_sql})").fetchone()[0]
    n_after = con.execute(f"SELECT COUNT(*) FROM ({after_sql})").fetchone()[0]
    if n_after != n_before:
        print(f"WARNING: the join changed the row count from {n_before:,} to {n_after:,} "
              f"({n_after / n_before:.1f}x). Each row may be counted more than once.")
        return False
    print(f"OK: row count unchanged ({n_before:,} rows).")
    return True


west_customers = "SELECT * FROM customers WHERE region = 'West'"

# The bad query, before averaging: West customers joined to their orders
fanned_out = """
    SELECT c.* FROM customers AS c
    JOIN orders AS o ON c.customer_id = o.customer_id
    WHERE c.region = 'West'
"""

# A safe join: orders counted per customer FIRST, so there's one row per customer to join to
safe_join = """
    SELECT c.*, o.n_orders FROM customers AS c
    JOIN (SELECT customer_id, COUNT(*) AS n_orders FROM orders GROUP BY customer_id) AS o
      ON c.customer_id = o.customer_id
    WHERE c.region = 'West'
"""

print("Bad query:");  check_join(con, west_customers, fanned_out)
print("Good query:"); check_join(con, west_customers, west_customers)
print("Safe join:");  _ = check_join(con, west_customers, safe_join)

My check counts the rows before and after a join. On the bad query it goes off: the join turns 2,097 West customers into 14,191 rows, almost 7 times as many, so customers are clearly being double counted. On the good query there's no join, so it stays quiet. It also stays quiet when I join to a table that already has one row per customer (orders counted up per customer first). So the rule isn't "never join." It's "a join shouldn't change what one row means." If this check ran on every query behind a dashboard, the \$1,240 would have been caught before it ever got near a slide.

## Phase 2: The Rows That Disappear (30 pts)

### Step 2.1: Introduce the unknown (5 pts)

In [ ]:
# About 15% of customers get a missing region
is_missing = rng.random(N) < 0.15
customers_df.loc[is_missing, "region"] = None
con.execute("CREATE OR REPLACE TABLE customers AS SELECT * FROM customers_df")

print(con.execute("SELECT COUNT(*) AS missing_region FROM customers WHERE region IS NULL").df())

I wiped out the region for about 15% of customers, to mimic people who signed up before the region field was required. That left 401 customers with no region.

### Step 2.2: Lose them, then find them (15 pts)

In [ ]:
lost = con.execute("SELECT COUNT(*) FROM customers WHERE region != 'West'").fetchone()[0]
found = con.execute("""
    SELECT COUNT(*) FROM customers
    WHERE region != 'West' OR region IS NULL
""").fetchone()[0]

print(f"WHERE region != 'West':                  {lost:,} rows")
print(f"WHERE region != 'West' OR region IS NULL: {found:,} rows")
print(f"Customers silently dropped:               {found - lost:,}")

`WHERE region != 'West'` gives back 777 customers. But what the analyst actually wanted was everyone outside the West, which is 1,178. The 401 missing are exactly the customers with no region.

Why it happens: if a customer's region is blank, SQL can't tell whether it's 'West' or not, so the comparison comes out as "unknown" instead of true or false, and `WHERE` only keeps rows where the answer is true.

This is a bigger deal than it looks. Every query that filters or groups by region quietly skips these customers, and nothing tells you it happened. If they're different from everyone else, say newer, smaller, or more likely to leave, then every regional number really only describes the older, more settled customers. Each region looks healthier than it is, the regional totals won't add up to the company total, and some regions could be off by more than others. And because the dropped customers never show up anywhere in the output, nobody would know to look for them.

### Step 2.3: Aggregate without collapsing (10 pts)

In [ ]:
with_region_avg = con.execute("""
    SELECT customer_id,
           region,
           revenue,
           AVG(revenue) OVER (PARTITION BY region)           AS region_avg,
           revenue - AVG(revenue) OVER (PARTITION BY region) AS above_region_avg
    FROM customers
    ORDER BY customer_id
""").df()

print(f"Rows in customers: {N:,}   rows in the result: {len(with_region_avg):,}")
with_region_avg.head(10)

This puts each customer's revenue right next to their region's average in a single query: 3,000 rows in, 3,000 rows out, no join and no subquery. GROUP BY would squash each region down to one row, but the window function keeps every customer and just adds their region's average beside them, so the director can see who's above and who's below. One thing to watch: the 401 customers with no region all get lumped together, so their "region average" doesn't belong to any real region.

## Phase 3: The Causal Question (25 pts)

### Step 3.1: The programme evaluation (12 pts)

In [ ]:
import statsmodels.api as sm      # for sm.add_constant and sm.OLS, as in Lab 10

def coef_on_d(y, d, controls):
    """Regress y on d plus every array in controls, with a constant; return d's coefficient."""
    X = sm.add_constant(np.column_stack([d] + controls))
    return sm.OLS(y, X).fit().params[1]


n = 10_000
TRUE_EFFECT = 50

# Existing account size drives BOTH which customers are in the West and their revenue
account_size = rng.normal(0, 1, n)
in_west = (rng.random(n) < 1 / (1 + np.exp(-1.5 * account_size))).astype(float)   # bigger accounts more likely West
programme_revenue = 800 + TRUE_EFFECT * in_west + 300 * account_size + rng.normal(0, 150, n)

naive = coef_on_d(programme_revenue, in_west, [])
adjusted = coef_on_d(programme_revenue, in_west, [account_size])

print(f"Share of customers in the West: {in_west.mean():.1%}")
print(f"{'estimate':<34}{'value':>8}{'truth':>8}")
print(f"{'Naive (West minus other regions)':<34}{naive:>8.1f}{TRUE_EFFECT:>8}")
print(f"{'Adjusted for account size':<34}{adjusted:>8.1f}{TRUE_EFFECT:>8}")

In this simulated world, I know the programme's real effect is \$50 per customer. Just comparing the West to everyone else gives \$374, more than seven times too high. That's because bigger accounts were more likely to end up in the West, and bigger accounts bring in more money whether or not they got the programme. So the simple comparison mixes up "West customers are bigger" with "the programme works." Once I control for account size, I'm comparing customers of about the same size, and the estimate drops to \$47, pretty close to the real \$50.

### Step 3.2: The control that ruins it (13 pts)

In [ ]:
# Support tickets are caused by BOTH the programme and revenue
tickets = 3 * in_west + 0.01 * programme_revenue + rng.normal(0, 1, n)

with_tickets = coef_on_d(programme_revenue, in_west, [account_size, tickets])

print(f"{'estimate':<40}{'value':>8}{'truth':>8}")
print(f"{'Adjusted for account size':<40}{adjusted:>8.1f}{TRUE_EFFECT:>8}")
print(f"{'Adjusted for account size + tickets':<40}{with_tickets:>8.1f}{TRUE_EFFECT:>8}")

Adding support tickets as a control moved the estimate from \$47 (close to the real \$50) all the way to −\$194, which would mean the programme loses money.

**To the director:** Support tickets sound like a reasonable way to measure engagement, but in our data they go up for two separate reasons. The programme itself creates tickets, since account managers push customers to reach out, and customers who spend more also file more tickets. So if we compare customers who have the same number of tickets, here's what happens: a programme customer got some of those tickets from the programme, while a non-programme customer with the same count must have gotten them by spending more. That means at every ticket level, the non-programme customers we're comparing against are the bigger spenders, and the programme looks worse than it really is. In short, "controlling for engagement" ended up stacking the comparison against the programme. My rule of thumb: don't control for anything the programme itself changes, or anything driven by the revenue we're trying to measure.

## Phase 4: AI Expansion — The Co-Pilot Era (20 pts)

**Status: AI usage AUTHORISED and REQUIRED.** Use the **P.R.I.M.E. framework**.

### Task 4.1: The query reviewer (12 pts)

In [ ]:
def review_query(con, sql):
    """The model's function from the P.R.I.M.E. prompt, then your corrections:
    returns a list of structured warnings for one SELECT."""
    ...

In [ ]:
# Run review_query on your Phase 1 and Phase 2 queries.

*Your answer here.*

### Task 4.2: The one-page writeup (8 pts)

**To:** Director  **Re:** West revenue per customer

**The right number is \$814.** You asked for average revenue per customer, so each customer should count once. The \$1,240 came from a query that joined customers to their orders before averaging, so each customer got counted once per order. Big customers place more orders, so they got counted many times, and that pushed the average up by about 54%.

**Why the two differed:** nobody made a math mistake. One query quietly changed what a row meant, from a customer to an order, and nothing caught it.

**Does the programme work?** Maybe a little, but much less than the regional gap makes it look. In a simulation where I know the real effect is \$50 per customer, comparing the West to other regions gives \$374, because bigger accounts were more likely to be in the West. Adjusting for account size gives \$47. Adding "support tickets" as a control sounds sensible but made it −\$194, because the programme itself changes how many tickets people file. To really know, we'd need to roll the programme out to randomly chosen customers, or at least compare customers of similar size.

**What I'd recommend before any number goes on a slide:**
1. Check that joins don't multiply rows when we're averaging per customer.
2. Make sure queries keep customers with a missing region instead of silently dropping 15% of them.
3. Have a second analyst rerun any number before it goes to the board.

### AI Transcript

Paste **your prompt exactly as you sent it** and the model's replies **exactly as they came back**, then one line on each change you made. Name the model and the date.

**Model and date:** [model], [date]

**Prompt:**

```text
[paste the prompt exactly as you sent it]
```

**Reply:**

[paste the reply exactly as it came back; put any code inside a ```python fence]

**Changes I made:**
- [one line per change]

## Before you submit

- [ ] Every heading above is still here, in order, with work under it.
- [ ] Every number you report is produced by a cell above it, and every claim has a text cell.
- [ ] The verification checkpoint(s) are met before anything built on them.
- [ ] Phase 4: the AI transcript is the exchange, not a summary; your prompt is in it as sent.
- [ ] Submit the Colab share link on Canvas (upload the `.ipynb` as a fallback).
- [ ] GitHub portfolio: *File › Save a copy in GitHub*, path `Problem Set 3/Econ_3916_PS3.ipynb`, with a short `README.md` in that folder.
- [ ] Run *Runtime › Restart and run all* once, top to bottom, before you hand in.